# GPT-2 Small (124M) treinado do zero

Notebook de treinamento sobre o FineWeb-Edu (`sample-10BT`) usando o tokenizer GPT-2. O checkpoint analisado registra 42 épocas lógicas, 10.500 steps e aproximadamente 5,51 bilhões de tokens processados.

Resultados finais: train loss 3,3080; validation loss 3,2915; validation perplexity 26,88; HellaSwag 27,68% em 10.042 exemplos.

In [ ]:
%pip install -q -U datasets tiktoken


In [ ]:
from dataclasses import dataclass
from itertools import islice
import inspect
import math
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from datasets import load_dataset

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN carregado dos Secrets do Kaggle.")
except Exception:
    print("HF_TOKEN não configurado; acesso público ao Hugging Face será usado.")


## Configuração e arquitetura do modelo



In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.l1 = torch.nn.Linear(config.n_embd, config.n_embd * 4)
        self.gelu = torch.nn.GELU("tanh")
        self.l2 = torch.nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        x = self.l1(x)
        x = self.gelu(x)
        x = self.l2(x)
        return x


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        #dhead = 64
        # key, query, value em batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # Flash Attention quando suportado pelo dispositivo/PyTorch.
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.ln1 = torch.nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.mlp = MLP(config)
        self.ln2 = torch.nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))  # residual stream livre de normalização
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config
        self.transformer = nn.ModuleDict(
            dict(
                wte=nn.Embedding(config.vocab_size, config.n_embd),
                wpe=nn.Embedding(config.block_size, config.n_embd),
                h=nn.ModuleList(Block(config) for _ in range(config.n_layer)),
                ln_f=nn.LayerNorm(config.n_embd),
            )
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # weight sharing scheme
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02

            if hasattr(module, "NANOGPT_SCALE_INIT"):
                std = (self.config.n_layer * 2) ** -(1 / 2)

            torch.nn.init.normal_(module.weight, mean=0.0, std=std)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def configure_optimizers(
        self,
        weight_decay,
        learning_rate,
        device,
        betas,
    ):
        param_dict = {pn: p for pn, p in self.named_parameters()}
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}

        # Tensores 2D ou maiores recebem weight decay; biases e LayerNorms não.
        decay_params = [p for _, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for _, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {"params": decay_params, "weight_decay": weight_decay},
            {"params": nodecay_params, "weight_decay": 0.0},
        ]

        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)

        print(
            f"num_decay_params={num_decay_params:,}, "
            f"num_nodecay_params={num_nodecay_params:,}"
        )

        fused_available = "fused" in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device == "cuda"
        print(
            f"using fused AdamW: {use_fused} "
            f"(fused_available={fused_available})"
        )

        optimizer = torch.optim.AdamW(
            optim_groups,
            lr=learning_rate,
            betas=betas,
            weight_decay=weight_decay,
            fused=use_fused,
        )
        return optimizer

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, (
            f"Cannot forward sequence of length {T}; "
            f"block size is only {self.config.block_size}"
        )

        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(idx)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.transpose(1, 2),
                targets.view(B, T),
            )

        return logits, loss


In [ ]:
#configurações do corpus em streaming.
FINEWEB_CONFIG = "sample-10BT"
VALIDATION_DOCUMENTS = 2_000
SHUFFLE_BUFFER_SIZE = 10_000
TOKENIZER_DOCUMENT_BATCH_SIZE = 32
TOKENIZER_THREADS = min(8, os.cpu_count() or 1)


class DataLoaderLite:
    """Loader infinito de blocos causais produzido a partir do FineWeb-Edu."""

    def __init__(
        self,
        B,
        T,
        split,
        validation_documents=VALIDATION_DOCUMENTS,
        seed=42,
    ):
        assert split in {"train", "val"}

        self.B = B
        self.T = T
        self.split = split
        self.seed = seed
        self.stream_epoch = 0

        #substitui o Tiny Shakespeare pelo FineWeb-Edu em streaming.
        base_dataset = load_dataset(
            "HuggingFaceFW/fineweb-edu",
            name=FINEWEB_CONFIG,
            split="train",
            streaming=True,
        )

        if split == "val":
            # Validação fixa: sempre os mesmos documentos, sem shuffle.
            self.dataset = base_dataset.take(validation_documents)
        else:
            # O treino nunca vê os documentos reservados para validação.
            self.dataset = base_dataset.skip(validation_documents)
            self.dataset = self.dataset.shuffle(
                seed=seed,
                buffer_size=SHUFFLE_BUFFER_SIZE,
            )

        self.enc = tiktoken.get_encoding("gpt2")
        self.eot_token = self.enc.eot_token
        self.reset()

    def _new_iterator(self):
        # Quando o stream completo terminar, muda a época interna do shuffle.
        if self.split == "train" and hasattr(self.dataset, "set_epoch"):
            self.dataset.set_epoch(self.stream_epoch)
        self.iterator = iter(self.dataset)

    def reset(self):
        """Reinicia o split; usado para medir sempre a mesma validação."""
        self.token_buffer = []
        self.current_position = 0
        self._new_iterator()

    def _fill_token_buffer(self, required_tokens):
        while len(self.token_buffer) - self.current_position < required_tokens:
            examples = list(
                islice(self.iterator, TOKENIZER_DOCUMENT_BATCH_SIZE)
            )

            if not examples:
                # Reinicia apenas ao consumir o stream selecionado por completo.
                self.stream_epoch += 1
                self._new_iterator()
                continue

            texts = [example["text"] for example in examples if example["text"]]
            documents = self.enc.encode_ordinary_batch(
                texts,
                num_threads=TOKENIZER_THREADS,
            )

            for document_tokens in documents:
                self.token_buffer.extend(document_tokens)
                self.token_buffer.append(self.eot_token)

    def next_batch(self):
        B, T = self.B, self.T
        required_tokens = B * T + 1
        self._fill_token_buffer(required_tokens)

        start = self.current_position
        end = start + required_tokens
        buf = torch.tensor(
            self.token_buffer[start:end],
            dtype=torch.long,
        )

        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)

        #avança B*T, preservando o último token como início do próximo bloco. Isso mantém o fluxo causal sem descartar um token.
        self.current_position += B * T

        # Compacta ocasionalmente o buffer para limitar o uso de RAM.
        if self.current_position >= 1_000_000:
            self.token_buffer = self.token_buffer[self.current_position :]
            self.current_position = 0

        return x, y


## HellaSwag

In [ ]:
def _prepare_hellaswag_choices(example, enc, block_size):
    context_tokens = enc.encode(example["ctx"])
    rows = []
    masks = []

    for ending in example["endings"]:
        # O espaço separa naturalmente o contexto da continuação.
        ending_tokens = enc.encode(" " + ending)
        tokens = context_tokens + ending_tokens
        completion_mask = [0] * len(context_tokens) + [1] * len(ending_tokens)

        # Preserva o final da sequência se ela exceder o contexto do modelo.
        if len(tokens) > block_size:
            overflow = len(tokens) - block_size
            tokens = tokens[overflow:]
            completion_mask = completion_mask[overflow:]

        rows.append(tokens)
        masks.append(completion_mask)

    max_length = max(len(row) for row in rows)
    token_tensor = torch.full(
        (len(rows), max_length),
        enc.eot_token,
        dtype=torch.long,
    )
    mask_tensor = torch.zeros(
        (len(rows), max_length),
        dtype=torch.float32,
    )

    for row_index, (row, mask) in enumerate(zip(rows, masks)):
        token_tensor[row_index, : len(row)] = torch.tensor(row)
        mask_tensor[row_index, : len(mask)] = torch.tensor(mask)

    return token_tensor, mask_tensor


@torch.inference_mode()
def evaluate_hellaswag(model, device, max_examples=500):
    #HellaSwag é lido em streaming para manter o notebook leve
    dataset = load_dataset(
        "Rowan/hellaswag",
        split="validation",
        streaming=True,
    )
    enc = tiktoken.get_encoding("gpt2")

    model.eval()
    correct = 0
    total = 0

    for example in dataset:
        if max_examples is not None and total >= max_examples:
            break

        # O campo label vem como string no dataset
        if example["label"] not in {"0", "1", "2", "3"}:
            continue

        tokens, completion_mask = _prepare_hellaswag_choices(
            example,
            enc,
            model.config.block_size,
        )
        tokens = tokens.to(device)
        completion_mask = completion_mask.to(device)

        # O padding está somente à direita. Posições reais não atendem ao padding
        # por causa da atenção causal, e a loss do padding é removida pela máscara.
        logits, _ = model(tokens[:, :-1])
        targets = tokens[:, 1:]
        target_mask = completion_mask[:, 1:]

        token_losses = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            reduction="none",
        ).view_as(targets)

        choice_losses = (token_losses * target_mask).sum(dim=1)
        choice_losses /= target_mask.sum(dim=1).clamp_min(1)

        prediction = int(torch.argmin(choice_losses).item())
        label = int(example["label"])
        correct += int(prediction == label)
        total += 1

        if total % 100 == 0:
            print(
                f"HellaSwag: {total} exemplos, "
                f"accuracy parcial={correct / total:.4f}"
            )

    model.train()
    return correct / max(total, 1), total


In [ ]:
from dataclasses import asdict
from google.colab import drive
import os
import torch

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/gpt2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def salvar_checkpoint(
    model,
    optim,
    nome,
    global_step,
    epoch,
    history,
    scaler=None,
):
    if not nome.endswith(".pt"):
        nome += ".pt"

    caminho = os.path.join(CHECKPOINT_DIR, nome)

    # Funciona também se o modelo estiver usando torch.compile
    base_model = getattr(model, "_orig_mod", model)

    torch.save(
        {
            "config": asdict(base_model.config),
            "model": base_model.state_dict(),
            "optimizer": optim.state_dict(),
            "global_step": global_step,
            "epoch": epoch,
            "history": history,
            "scaler": (
                scaler.state_dict()
                if scaler is not None
                else None
            ),
        },
        caminho,
    )

    print(f"Checkpoint salvo em: {caminho}")


def carregar_checkpoint(nome, scaler=None):
    if not nome.endswith(".pt"):
        nome += ".pt"

    caminho = os.path.join(CHECKPOINT_DIR, nome)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    checkpoint = torch.load(
        caminho,
        map_location=device,
        weights_only=True,
    )

    config = GPTConfig(**checkpoint["config"])
    model = GPT(config).to(device)
    model.load_state_dict(checkpoint["model"])

    optim = model.configure_optimizers(
        weight_decay=0.1,
        learning_rate=max_lr,
        betas=(0.9, 0.95),
        device=device,
    )
    optim.load_state_dict(checkpoint["optimizer"])

    if scaler is not None and checkpoint["scaler"] is not None:
        scaler.load_state_dict(checkpoint["scaler"])

    model.train()

    return (
        model,
        optim,
        checkpoint["global_step"],
        checkpoint["epoch"],
        checkpoint["history"],
    )

## Parâmetros de execução

Em streaming, uma passagem completa pelo `sample-10BT` seria enorme. Portanto,
uma **época lógica** é definida por `STEPS_PER_EPOCH` atualizações do otimizador.
O stream de treino continua de onde parou entre épocas; ele não volta ao início.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


total_batch_size = 1 << 19  # tokens por step, igual no paper do gpt2
#total_batch_size = 1 << 15  # tokens por step, igual no paper do gpt2
B, T = 32, 1024 #Tentar maximizar a vram
assert total_batch_size % (B * T) == 0

#total_batch_size = (500000 + (B * T + 1))/(B * T) * (B * T)
#total_batch_size = int(total_batch_size)

grad_accumulation_steps = total_batch_size // (B * T)

#10B tokens

# ALTERAÇÃO: épocas lógicas e validação configuráveis para o Kaggle.
STEPS_PER_EPOCH = 250
#NUM_EPOCHS = 5 #evaluation
NUM_EPOCHS = int(10e9  // ((1 << 19) * STEPS_PER_EPOCH)) # 10B tokens
VAL_STEPS = 20

NOME_MODELO = 'meu-gpt2-small'

model = GPT(GPTConfig()).to(device)

In [ ]:
NUM_EPOCHS

In [ ]:
grad_accumulation_steps, B*T, total_batch_size

In [ ]:
train_loader = DataLoaderLite(B, T, split="train")
val_loader = DataLoaderLite(B, T, split="val")

print(
    f"grad_accumulation_steps={grad_accumulation_steps}, "
    f"tokens_per_epoch={total_batch_size * STEPS_PER_EPOCH:,}"
)


In [ ]:
model = torch.compile(model)

In [ ]:
max_lr = 6e-4
min_lr = max_lr * 0.1
warmup_steps = 10
max_steps = NUM_EPOCHS * STEPS_PER_EPOCH

# Cosine decay com warmup, como no paper do GPT2
def get_lr(step):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step >= max_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + (max_lr - min_lr) * (
        1 + math.cos(math.pi * decay_ratio)
    ) * 0.5


optim = model.configure_optimizers(
    weight_decay=0.1,
    learning_rate=max_lr,
    betas=(0.9, 0.95),
    device=device,
)


@torch.inference_mode()
def estimate_validation_loss(model, val_loader, steps):
    """Mede sempre os mesmos blocos reservados do FineWeb-Edu."""
    model.eval()
    val_loader.reset()
    losses = []

    for _ in range(steps):
        x, y = val_loader.next_batch()
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        _, loss = model(x, y)
        losses.append(loss.detach())

    mean_loss = torch.stack(losses).mean().item()
    model.train()
    return mean_loss


In [ ]:
# Loop de treinamento principal, com validação ao final de cada época lógica

tps_list = []
history = []
global_step = 0

torch.set_float32_matmul_precision('high')

for epoch in range(NUM_EPOCHS):
    epoch_train_loss = 0.0
    epoch_start = time.time()
    model.train()

    for step_in_epoch in range(STEPS_PER_EPOCH):
        optim.zero_grad(set_to_none=True)
        t0 = time.time()
        loss_accum = torch.zeros((), device=device)

        #batch
        for micro_step in range(grad_accumulation_steps):
            x, y = train_loader.next_batch()
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            #with torch.autocast(device_type=device, dtype=torch.float16): ruim infelizmente
            logits, loss = model(x, y)
            loss_accum += loss.detach()
            # Mantém a média correta ao acumular gradientes
            (loss / grad_accumulation_steps).backward()

        loss_accum /= grad_accumulation_steps

        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        lr = get_lr(global_step)
        for param_group in optim.param_groups:
            param_group["lr"] = lr

        optim.step()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.time() - t0
        tokens_per_second = total_batch_size / elapsed
        tps_list.append(tokens_per_second)

        train_loss_value = loss_accum.item()
        epoch_train_loss += train_loss_value

        if step_in_epoch % 10 == 0:
            print(
                f"epoch={epoch + 1}/{NUM_EPOCHS} "
                f"step={step_in_epoch:04d}/{STEPS_PER_EPOCH} "
                f"global_step={global_step:05d} "
                f"train_loss={train_loss_value:.4f} "
                f"lr={lr:.2e} "
                f"tps={tokens_per_second:.2f} "
                f"norm={float(norm):.4f}"
            )

        global_step += 1

    val_loss = estimate_validation_loss(model, val_loader, VAL_STEPS)
    train_loss = epoch_train_loss / STEPS_PER_EPOCH
    val_perplexity = math.exp(min(val_loss, 20))
    epoch_minutes = (time.time() - epoch_start) / 60

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_perplexity": val_perplexity,
        }
    )

    print(
        f"\nFim da época {epoch + 1}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"val_ppl={val_perplexity:.2f}, "
        f"tempo={epoch_minutes:.2f} min\n"
    )

    salvar_checkpoint(
      model=model,
      optim=optim,
      nome=NOME_MODELO,
      global_step=global_step,
      epoch=epoch,
      history=history,
      scaler=None,
    )

print(f"Average tokens per second: {sum(tps_list) / len(tps_list):.2f}")


In [ ]:
#Avaliação do modelo no HellaSwag, com limite de exemplos para teste rápido.
HELLASWAG_MAX_EXAMPLES = None
hellaswag_accuracy, hellaswag_examples = evaluate_hellaswag(
    model,
    device,
    max_examples=HELLASWAG_MAX_EXAMPLES,
)

print(
    f"HellaSwag accuracy={hellaswag_accuracy:.4f} "
    f"em {hellaswag_examples} exemplos"
)


In [ ]:
tokens_treinados = global_step * total_batch_size

print(f"{tokens_treinados / 1e9:.2f} bilhões de tokens")

In [ ]:
#generate
enc = tiktoken.get_encoding("gpt2")
text = "Hi! I'm a"
input_ids = enc.encode(text)
# O modelo espera o formato (B, T)
inputs = torch.tensor(
    input_ids,
    dtype=torch.long,
    device=device,
).unsqueeze(0)

max_new_tokens = 32
temperature = 0.8
top_k = 25

# Compatível com modelos que passaram por torch.compile
base_model = getattr(model, "_orig_mod", model)
block_size = base_model.config.block_size

model.eval()

with torch.inference_mode():
    for _ in range(max_new_tokens):

        # Não ultrapassa o contexto máximo do modelo
        context = inputs[:, -block_size:]

        # logits: (B, T, vocab_size)
        logits, _ = model(context)

        next_token_logits = logits[:, -1, :]

        next_token_logits = next_token_logits / temperature

        # Considera somente os top_k tokens mais prováveis
        k = min(top_k, next_token_logits.size(-1))
        values, _ = torch.topk(next_token_logits, k)

        next_token_logits[
            next_token_logits < values[:, [-1]]
        ] = -float("inf")

        probabilities = F.softmax(
            next_token_logits,
            dim=-1,
        )

        # Sorteia o próximo token
        next_token = torch.multinomial(
            probabilities,
            num_samples=1,
        )

        inputs = torch.cat(
            (inputs, next_token),
            dim=1,
        )

        if next_token.item() == enc.eot_token:
            break

generated_text = enc.decode(inputs[0].tolist())

print(generated_text)



## Curvas de treinamento
A célula abaixo usa a lista `history` e salva o gráfico em PNG.

In [ ]:
import matplotlib.pyplot as plt

epochs = [item["epoch"] for item in history]
train_losses = [item["train_loss"] for item in history]
val_losses = [item["val_loss"] for item in history]
val_perplexities = [item["val_perplexity"] for item in history]

plt.style.use("seaborn-v0_8-whitegrid")
fig, (ax_loss, ax_ppl) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GPT-2 Small (124M) — evolução do treinamento", fontsize=16, fontweight="bold")

ax_loss.plot(epochs, train_losses, label="Treino", linewidth=2)
ax_loss.plot(epochs, val_losses, label="Validação", linewidth=2)
ax_loss.set(title="Loss por época", xlabel="Época", ylabel="Cross-entropy loss")
ax_loss.legend()

ax_ppl.plot(epochs, val_perplexities, color="#7c3aed", linewidth=2)
ax_ppl.set_yscale("log")
ax_ppl.set(title="Perplexidade de validação", xlabel="Época", ylabel="Perplexidade (escala log)")

final = history[-1]
ax_loss.annotate(f"{final['val_loss']:.3f}", (final["epoch"], final["val_loss"]))
ax_ppl.annotate(f"{final['val_perplexity']:.2f}", (final["epoch"], final["val_perplexity"]))

fig.tight_layout()
fig.savefig("training_curves.png", dpi=180, bbox_inches="tight")
plt.show()